In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pickle

In [2]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

In [3]:
# pd.read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet')

In [4]:
#df.duration.describe(percentiles=[0.95,0.98,0.99])

In [5]:
# what kind of categorical and numerical variable we will use

In [6]:
# we can think of weeked, number of passenger, 
categorical = ['PULocationID','DOLocationID']
numerical = ['trip_distance']

In [7]:
# PU and DU are categorical. That's why he used one hot encoding to process that. 
# This algorithm what actually does is create one variable for each category 
# and this variable is discrete with values in {0, 1}, i.e. Binary.
# does not work on numerical value
# So that regression is actually a multivariate regression and the values it takes are numerical.

In [8]:
# df[categorical] = df[categorical].astype(str)
# df[categorical].dtypes

In [9]:
# vectorise a dictionary, turn a dictionary into a vector
# we need a dictionary rn we have a dataframe, turn the datagrame into bunch of dict
# turn each row into a dictionary

In [10]:
# train_dicts = df[categorical + numerical].to_dict(orient = 'records')
# train_dicts

In [11]:
# dv = DictVectorizer()
# X_train = dv.fit_transform(train_dicts)

In [12]:
# target = 'duration'
# y_train = df[target].values
# y_train

In [13]:
# lr = LinearRegression()
# lr.fit(X_train, y_train)

In [14]:
# y_pred = lr.predict(X_train)

In [15]:
# sns.histplot(y_pred, color='blue', label='Prediction', kde=True, alpha=0.5)
# sns.histplot(y_train, color='red', label='Actual', kde=True, alpha=0.5)

# plt.legend()
# plt.title("Comparison of Predictions vs Actual Values")
# plt.xlabel("Target Values")
# plt.ylabel("Frequency")
# plt.show()

In [16]:
# compute Root mean squared term on training
# mean_squared_error(y_train, y_pred, squared=False)

In [17]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)
    df['duration'] = (df.lpep_dropoff_datetime - df.lpep_pickup_datetime)
    df['duration'] = df.duration.apply(lambda td:td.seconds / 60)
    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID','DOLocationID']

    df[categorical] = df[categorical].astype(str)
    return df

In [18]:
# january train
df_train = read_dataframe('./data/green_tripdata_2021-01.parquet')
# feb validation
df_val = read_dataframe('./data/green_tripdata_2021-02.parquet')

In [19]:
len(df_train), len(df_val)

(73908, 61921)

In [20]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

In [21]:
categorical = ['PULocationID','DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient = 'records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient = 'records')
X_val  = dv.transform(val_dicts)

In [22]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [23]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

# compute Root mean squared term on training
mean_squared_error(y_val, y_pred, squared=False)


10.473871054242926

In [24]:
# on validation

In [25]:
# our model is wrong on average 9 mins

In [26]:
# try lasso
lr = Lasso(alpha=0.05)
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

mean_squared_error(y_val, y_pred, squared=False)

11.439943687389436

In [27]:
# try Ridge
lr = Ridge()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

mean_squared_error(y_val, y_pred, squared=False)

10.86075339035955

In [28]:
# adding this featur e help we reduce the error significantly, we can predict better

In [29]:
categorical = ['PU_DO']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient = 'records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient = 'records')
X_val  = dv.transform(val_dicts)

In [30]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [31]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

# compute Root mean squared term on training
mean_squared_error(y_val, y_pred, squared=False)

7.4793657829373155

In [237]:
# save the model
with open('models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)